In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip -oq "/content/drive/MyDrive/archive (9).zip" -d "/content/inbreast"


In [4]:
import os
import pandas as pd
import numpy as np
import cv2

import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from tensorflow.keras import layers, models

In [7]:
import pandas as pd

csv_path = "/content/INbreast.csv"
df = pd.read_csv(csv_path, sep=';')
print(df.columns)
df.head()


Index(['Patient ID', 'Patient age', 'Laterality', 'View', 'Acquisition date',
       'File Name', 'ACR', 'Bi-Rads'],
      dtype='object')


,Patient ID,Patient age,Laterality,View,Acquisition date,File Name,ACR,Bi-Rads
0,removed,removed,R,CC,201001,22678622,4,1
1,removed,removed,L,CC,201001,22678646,4,3
2,removed,removed,R,MLO,201001,22678670,4,1
3,removed,removed,L,MLO,201001,22678694,4,3
4,removed,removed,R,CC,201001,22614074,2,5


In [8]:
import os
import numpy as np

benign = ['1', '2']
malignant = ['4', '4a', '4b', '4c', '5', '6']
df = df[df['Bi-Rads'].astype(str).isin(benign + malignant)]
df['label'] = df['Bi-Rads'].astype(str).apply(lambda x: 1 if x in malignant else 0)

dicom_dir = "/content/inbreast/INbreast Release 1.0/AllDICOMs"
dcm_files = os.listdir(dicom_dir)
def match_dcm(x):
    return next((f for f in dcm_files if f.startswith(str(x) + '_')), None)
df['matched_file'] = df['File Name'].apply(match_dcm)
df.dropna(subset=['matched_file'], inplace=True)
df['image_path'] = df['matched_file'].apply(lambda x: os.path.join(dicom_dir, x))

# Upsample class 1
df_benign = df[df['label'] == 0]
df_malignant = df[df['label'] == 1].sample(len(df_benign), replace=True, random_state=42)
df = pd.concat([df_benign, df_malignant]).sample(frac=1, random_state=42).reset_index(drop=True)

print("✅ Final label distribution:")
print(df['label'].value_counts())


✅ Final label distribution:
label
1    287
0    287
Name: count, dtype: int64


/tmp/ipython-input-8-1839012501.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['Bi-Rads'].astype(str).apply(lambda x: 1 if x in malignant else 0)
/tmp/ipython-input-8-1839012501.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['matched_file'] = df['File Name'].apply(match_dcm)
/tmp/ipython-input-8-1839012501.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-doc

In [17]:
!pip install -q pydicom


In [22]:
import cv2
import pydicom
from tqdm import tqdm

IMG_SIZE = 384
X, y = [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        dcm = pydicom.dcmread(row['image_path'])
        img = dcm.pixel_array.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min())
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        X.append(img)
        y.append(row['label'])
    except:
        continue

X = np.array(X)[..., np.newaxis]
y = np.array(y)
print("✅ Image shape:", X.shape)


100%|██████████| 574/574 [01:13<00:00,  7.78it/s]


✅ Image shape: (574, 384, 384, 1)


In [23]:
from sklearn.model_selection import train_test_split
import tensorflow as tf

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

BATCH_SIZE = 8
def augment(x, y):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    x = tf.image.random_brightness(x, 0.1)
    x = tf.image.random_contrast(x, 0.9, 1.1)
    return x, y

def tf_ds(X, y, aug=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y)).map(lambda x, y: (tf.cast(x, tf.float32), y))
    if aug:
        ds = ds.map(augment)
    return ds.shuffle(256).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = tf_ds(X_train, y_train, aug=True)
val_ds = tf_ds(X_val, y_val)


In [24]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

# Input: 384x384x1 → tile to 3 channels
input_layer = tf.keras.Input(shape=(384, 384, 1))
x = tf.keras.layers.Concatenate()([input_layer, input_layer, input_layer])

# Load DenseNet backbone
base_model = DenseNet121(include_top=False, weights='imagenet', input_tensor=x)
base_model.trainable = True

# Custom classifier head
x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=input_layer, outputs=output)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 384, 384,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 384, 384,  │          0 │ input_layer_3[0]… │
│ (Concatenate)       │ 3)                │            │ input_layer_3[0]… │
│                     │                   │            │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_4    │ (None, 390, 390,  │          0 │ concatenate_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 192, 192,  │      9,408 │ zero_padding2d_4… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 192, 192,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 192, 192,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_5    │ (None, 194, 194,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 96, 96,    │          0 │ zero_padding2d_5… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 96, 96,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 96, 96,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 96, 96,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 96, 96,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 96, 96,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 96, 96,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 96, 96,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 96, 96,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 7,168,833 (27.35 MB)

 Trainable params: 7,085,185 (27.03 MB)

 Non-trainable params: 83,648 (326.75 KB)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=6, mode='max', restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_auc', mode='max', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)


Epoch 1/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 20s/step - accuracy: 0.4968 - auc: 0.5045 - loss: 0.8770 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1338s 21s/step - accuracy: 0.4975 - auc: 0.5052 - loss: 0.8755 - val_accuracy: 0.5043 - val_auc: 0.6198 - val_loss: 0.7675 - learning_rate: 1.0000e-04
Epoch 2/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 20s/step - accuracy: 0.6531 - auc: 0.6871 - loss: 0.6443 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1274s 21s/step - accuracy: 0.6528 - auc: 0.6872 - loss: 0.6445 - val_accuracy: 0.6261 - val_auc: 0.6382 - val_loss: 0.6645 - learning_rate: 1.0000e-04
Epoch 3/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 19s/step - accuracy: 0.7319 - auc: 0.8107 - loss: 0.5352 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1214s 21s/step - accuracy: 0.7316 - auc: 0.8106 - loss: 0.5354 - val_accuracy: 0.6348 - val_auc: 0.6487 - val_loss: 0.6805 - learning_rate: 1.0000e-04
Epoch 4/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 19s/step - accuracy: 0.7831 - auc: 0.8531 - loss: 0.4856 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1170s 20s/step - accuracy: 0.7831 - auc: 0.8530 - loss: 0.4855 - val_accuracy: 0.6000 - val_auc: 0.7560 - val_loss: 0.5698 - learning_rate: 1.0000e-04
Epoch 5/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 19s/step - accuracy: 0.8058 - auc: 0.8865 - loss: 0.4336 

...

58/58 ━━━━━━━━━━━━━━━━━━━━ 1138s 20s/step - accuracy: 0.9224 - auc: 0.9749 - loss: 0.2049 - val_accuracy: 0.8783 - val_auc: 0.9629 - val_loss: 0.2844 - learning_rate: 2.5000e-05
Epoch 10/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.9329 - auc: 0.9675 - loss: 0.2247 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1137s 19s/step - accuracy: 0.9326 - auc: 0.9675 - loss: 0.2247 - val_accuracy: 0.8435 - val_auc: 0.9673 - val_loss: 0.4217 - learning_rate: 2.5000e-05
Epoch 11/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.9550 - auc: 0.9918 - loss: 0.1374 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1142s 20s/step - accuracy: 0.9549 - auc: 0.9917 - loss: 0.1375 - val_accuracy: 0.9217 - val_auc: 0.9817 - val_loss: 0.2038 - learning_rate: 1.2500e-05
Epoch 12/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.9400 - auc: 0.9792 - loss: 0.2070 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1114s 19s/step - accuracy: 0.9399 - auc: 0.9791 - loss: 0.2070 - val_accuracy: 0.9130 - val_auc: 0.9876 - val_loss: 0.1694 - learning_rate: 1.2500e-05
Epoch 13/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 1111s 19s/step - accuracy: 0.9610 - auc: 0.9917 - loss: 0.1317 - val_accuracy: 0.9304 - val_auc: 0.9849 - val_loss: 0.1768 - learning_rate: 1.2500e-05
Epoch 14/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 1119s 19s/step - accuracy: 0.9277 - auc: 0.9748 - loss: 0.1890 - val_accuracy: 0.9391 - val_auc: 0.9870 - val_loss: 0.1564 - learning_rate: 6.2500e-06
Epoch 15/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 1159s 19s/step - accuracy: 0.9647 - auc: 0.9959 - loss: 0.1021 - val_accuracy: 0.9217 - val_auc: 0.9855 - val_loss: 0.1738 - learning_rate: 6.2500e-06
Epoch 16/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.9532 - auc: 0.9843 - loss: 0.1485 

58/58 ━━━━━━━━━━━━━━━━━━━━ 1110s 19s/step - accuracy: 0.9529 - auc: 0.9842 - loss: 0.1492 - val_accuracy: 0.9304 - val_auc: 0.9885 - val_loss: 0.1406 - learning_rate: 6.2500e-06
Epoch 17/30
 2/58 ━━━━━━━━━━━━━━━━━━━━ 17:18 19s/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.0575  